# Fundamentals 11 - Multi-Agentic System API

Sistema multi-agente con solver determinista, judge determinista y reviewer LM opcional. `runtime(provider="auto")` decide backend; los agentes/frameworks obedecen ese runtime.


In [ ]:
import os
import agentic_systems as toolkit
PRETTY = False
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=8, max_turns=8)
local_runtime = toolkit.runtime(provider="python-direct", model="local-python", region="local", scheduler=scheduler)
lm_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
system = toolkit.AgenticSystem(model=lm_runtime.model_id or "local-python", region=lm_runtime.region_name or "local", runtime=lm_runtime)
toolkit.show({"local_runtime": local_runtime.describe(), "lm_runtime": lm_resolution, "lm_available": lm_available, "force_local_only": force_local_only})


In [ ]:
USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42
toolkit.show({"prompt": USER_PROMPT, "numbers": NUMBERS, "expected": EXPECTED})


## Par?metros de `RunPolicy`

`RunPolicy` declara c?mo debe comportarse una ejecución antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparaci?n, controla trazas y hace que el resultado sea evaluable.

| Par?metro | Qu? controla | Uso recomendado |
|---|---|---|
| `max_turns` | N?mero m?ximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | N?mero m?ximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | L?mite de tokens del modelo cuando el provider lo soporta. | ?til en providers LM; puede quedar `None` en `python-direct`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de selecci?n de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explícito cuando quieres forzar una tool. |
| `repair` | Permite reparaci?n autom?tica de salidas o tool calls inv?lidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | M?ximo de intentos de reparaci?n. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Qu? hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explícito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnóstico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para enseñar API y evitar ambig?edad. |


In [ ]:
@toolkit.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b
    x2 = x1 - c
    x3 = x2 * d
    x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}
@toolkit.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}
toolkit.show({"tools": [solve_arithmetic.name, judge_result.name]})


## 1) Declarar policy y contratos arriba

Los `run(...)` quedan limpios porque la intenci?n vive en `RunPolicy` y `AgentContract`.


In [ ]:
single_tool_policy = toolkit.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
solve_contract = toolkit.AgentContract(must_call=["solve_arithmetic"], tool_expectation=toolkit.expect.exactly("solve_arithmetic"), completion="when_required_tools_satisfied")
judge_contract = toolkit.AgentContract(must_call=["judge_result"], tool_expectation=toolkit.expect.exactly("judge_result"), completion="when_required_tools_satisfied")
toolkit.show({"policy": single_tool_policy.model_dump(mode="json"), "solve_contract": solve_contract.model_dump(mode="json"), "judge_contract": judge_contract.model_dump(mode="json")})


## 2) Crear agentes deterministas y LM opcional


In [ ]:
@toolkit.tool
def record_review(summary: str) -> dict:
    """Registra una revisi?n LM como evidencia estructurada."""
    return {"summary": summary}

solver = toolkit.agent(name="deterministic_solver", instructions="Resuelve n?meros estructurados.", tools=[solve_arithmetic], engine="python-direct", runtime=local_runtime, contract=solve_contract, policy=single_tool_policy)
judge = toolkit.agent(name="deterministic_judge", instructions="Valida resultado estructurado.", tools=[judge_result], engine="python-direct", runtime=local_runtime, contract=judge_contract, policy=single_tool_policy)
reviewer = system.agent(name="lm_reviewer", instructions="Explica la evidencia sin cambiar n?meros.", tools=[record_review], runtime=lm_runtime, policy=toolkit.RunPolicy.for_mode("eval"))
toolkit.show({"solver": solver.info(), "judge": judge.info(), "reviewer": reviewer.info()})


## 3) Ejecutar sistema multi-agente


In [ ]:
solve = solver.run({"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}})
judgement = judge.run({"tool": "judge_result", "input": {"result": solve.data["result"], "expected": EXPECTED}})
review = None
if lm_available:
    review = reviewer.run(str({"solution": solve.data, "judge": judgement.data}))
else:
    toolkit.show({"status": "skipped", "reason": lm_resolution["reason"]}, title="LM reviewer saltado")

final = {
    "procedimiento": solve.data["procedure"],
    "resultado_final": solve.data["result"],
    "judge": judgement.data,
    "lm_review": review.text if review else None,
}
system_result = toolkit.compose_result(
    text="Sistema multi-agente ejecutado.",
    data=final,
    results=[solve, judgement, review],
    mode="multi-agentic-system",
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution},
)
lineage = system_result.lineage(name="fundamentals.multi_agentic_system", question=USER_PROMPT, goal="Explicar solver + judge + reviewer.")
toolkit.human_result(system_result, title="Human result - Multi-Agentic System", pretty=PRETTY, show_lineage=True, lineage=lineage)

## Coverage API


In [ ]:
toolkit.show({"notebook": "11_multi_agentic_system_api.ipynb", "api_coverage": ["runtime(provider='auto')", "python-direct agents", "AgenticSystem.agent", "AgentContract", "RunPolicy", "compose_result", "RunResult.lineage", "human_result"]})


## Símbolos API explicados

Este notebook se alinea con `docs/API.md` y enseña estos símbolos públicos:

- `AgenticSystem`: Orquestaci?n multi-agente nativa.
- `toolkit.compose_result`: Consolidaci?n de resultados de varios agentes.
- `RuntimeConfig / SchedulerConfig`: Runtime y límites compartidos por el sistema.
- `AgentContract / RunPolicy`: Contratos consistentes entre agentes.

